## SDK 0.2.1 설치 / Install SDK 0.2.1
첫 셀은 SDK 0.2.1이 없을 때만 고정 wheel을 설치합니다(네트워크 필요). 설치된 경우 건너뜁니다. 결과 전송은 별도 선택이며 기본 False입니다.
The first cell installs the pinned wheel only if SDK 0.2.1 is absent (network required). If 0.2.1 is already installed, installation is skipped. Publishing results is a separate choice, default False.


In [ ]:
# Install only when this SDK version is absent. No result publication.
import importlib
import subprocess
import sys

try:
    import luna_repro
    current_sdk = getattr(luna_repro, "__version__", None)
except ModuleNotFoundError as error:
    if error.name != "luna_repro":
        raise
    current_sdk = None

if current_sdk != "0.2.1":
    subprocess.check_call([sys.executable, "-m", "pip", "install",
        "https://github.com/lunalab-ai/luna-repro-sdk/releases/download/v0.2.1/luna_repro-0.2.1-py3-none-any.whl#sha256=86a1a5fbe26c203241676cd97e0bb37a49d00d0a6b82cb951ba5f1c0bae40efa"])
    if current_sdk is not None:
        raise RuntimeError("SDK updated. Restart the runtime/kernel, then Run all again. SDK 업데이트 후 런타임을 재시작하고 다시 실행하세요.")
    importlib.invalidate_caches()
    import luna_repro
assert luna_repro.__version__ == "0.2.1"
print("LUNA Repro SDK", luna_repro.__version__)


# 마지막 셀에서 결과 등록 / Final-cell registration
먼저 SDK ZIP 또는 로컬 SDK 저장소 폴더를 프로젝트 환경에 설치하세요. 이 노트북은 합성 CPU 결과를 계산하며 연구 성능을 나타내지 않습니다.
Install the SDK ZIP/local repository first. This example computes synthetic CPU results, not research performance.
기본 설정은 로컬 저장만 수행합니다. 실제 전송은 HTTPS endpoint와 세션 token을 설정하고 마지막 `publish()` 셀을 실행하세요.

In [ ]:
# 기존 연구 코드에 해당하는 부분 / Your existing research computation
import math
import random
rng = random.Random(20261011)
reference = [math.sin(i / 12) for i in range(120)]
prediction = [v + rng.gauss(0, 0.25) for v in reference]
metrics = {'RMSE': math.sqrt(sum((a-b)**2 for a,b in zip(reference, prediction))/len(reference)), 'samples': len(reference)}
sample_rows = [[i, reference[i], prediction[i]] for i in range(10)]
print(metrics)

플랫폼의 실제 프로젝트 UUID로 바꾸세요. 아래 UUID는 합성 오프라인 예제용입니다. 토큰은 노트북에 적지 말고 `getpass.getpass()`로 `LUNA_REPRO_TOKEN` 환경 변수에 설정하세요. `LUNA_REPRO_ENDPOINT`는 실제 플랫폼 HTTPS 주소입니다.
Replace the example UUID with your platform project. Keep the token in the session environment only, never in source or outputs.

In [ ]:
# 마지막에 추가하는 셀 / Add this cell at the end
import os
import luna_repro as repro
page = repro.page(os.environ.get('LUNA_REPRO_PROJECT', '12345678-1234-5678-1234-567812345678'), '합성 신호 결과 / Synthetic signal', evidence={'origin': 'synthetic', 'execution_status': 'completed', 'evaluation_status': 'completed'})
page.section('quality', '정량 결과 / Metrics', source={'file': 'last_cell.ipynb', 'cell_id': 'research-computation'}).metrics(metrics).table(['index', 'reference', 'prediction'], sample_rows)
page.save('.luna-repro/last-cell-result.json')

In [ ]:
# 명시적 실제 전송: 연결 설정 후 아래 두 줄의 주석을 해제합니다.
# Explicit transmission: configure endpoint/token, then uncomment.
# receipt = page.publish()
# print(receipt.get('url') or receipt['message'])